# 27. Self-trained ablation with full Optuna tuning

Two-step ablation study to fairly compare self-trained co-training vs LG-CoTrain:

**Step 1: Generate optimized pseudo-labels**
- For each (event, budget, seed), train a BERTweet teacher using the Optuna-tuned
  hyperparameters loaded from `results/bertweet/supervised/optuna-tuned/`
- Run inference on the unlabeled pool to generate pseudo-labels
- Save to `data/pseudo-labelled/self-trained-optuna/`

**Step 2: Run Optuna on LG-CoTrain with self-trained pseudo-labels**
- Uses `lg_cotrain.optuna_per_experiment` with `--pseudo-label-source self-trained-optuna`
- Tunes the same 6 hyperparameters as the LG-CoTrain Optuna study (lr, batch_size,
  cotrain_epochs, finetune_patience, weight_decay, warmup_ratio)
- 10 trials per (event, budget, seed) study

This ensures full consistency with the LG-CoTrain Optuna study (results/bertweet/optuna/wge7).
The only difference is the pseudo-label source: GPT-4o vs self-trained BERTweet.

## Results paths

```
data/pseudo-labelled/self-trained-optuna/          <- Step 1 output
results/bertweet/ablation/self-trained/optuna/     <- Step 2 output
```

## Run order

The notebook is organized budget-by-budget so you can run incrementally and
analyze partial results before deciding whether to continue.

## Setup

In [ ]:
import sys
import os
import time
import json
import subprocess
import warnings
from pathlib import Path
from collections import defaultdict

# Suppress HuggingFace model load reports and warnings
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message=".*emoji is not installed.*")
warnings.filterwarnings("ignore", message=".*not sharded.*")

import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

def _find_repo_root(marker: str = "lg_cotrain") -> Path:
    for candidate in [Path().resolve()] + list(Path().resolve().parents):
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Cannot find repo root: no ancestor directory contains '{marker}/'")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
print(f"Python:    {sys.version.split()[0]}")

In [ ]:
import torch
print(f"torch version:     {torch.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"CUDA device count: {N_GPUS}")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    free_gb, total_gb = (x / 1024**3 for x in torch.cuda.mem_get_info(i))
    print(f"  cuda:{i}  {props.name}  total={total_gb:.1f} GB  free={free_gb:.1f} GB")

In [ ]:
EVENTS = [
    "kaikoura_earthquake_2016",
    "canada_wildfires_2016",
    "cyclone_idai_2019",
    "hurricane_florence_2018",
    "hurricane_maria_2017",
    "california_wildfires_2018",
    "hurricane_dorian_2019",
    "kerala_floods_2018",
    "hurricane_harvey_2017",
    "hurricane_irma_2017",
]
BUDGETS = [5, 10, 25, 50]
SEEDS = [1, 2, 3]
MODEL_NAME = "vinai/bertweet-base"
DATA_ROOT = repo_root / "data"
N_OPTUNA_TRIALS = 10

# Paths
PSEUDO_LABEL_DIR = DATA_ROOT / "pseudo-labelled" / "self-trained-optuna"
SUP_OPTUNA_DIR = repo_root / "results" / "bertweet" / "supervised" / "optuna-tuned"
OPTUNA_STORAGE = repo_root / "results" / "bertweet" / "ablation" / "self-trained" / "optuna"

# Reference results for comparison
LG_COTRAIN_OPTUNA = repo_root / "results" / "bertweet" / "optuna" / "wge7"

print(f"Events:          {len(EVENTS)}")
print(f"Budgets:         {BUDGETS}")
print(f"Seeds:           {SEEDS}")
print(f"Optuna trials:   {N_OPTUNA_TRIALS}")
print(f"Model:           {MODEL_NAME}")
print(f"Num GPUs:        {N_GPUS}")
print()
print(f"Pseudo-label output:  {PSEUDO_LABEL_DIR}")
print(f"Sup. Optuna params:   {SUP_OPTUNA_DIR}")
print(f"Optuna storage:       {OPTUNA_STORAGE}")

In [ ]:
# Load supervised Optuna params for teacher generation
sup_params = {}
for event in EVENTS:
    for budget in BUDGETS:
        for seed in SEEDS:
            fp = SUP_OPTUNA_DIR / event / f"{budget}_set{seed}" / "trials_10" / "best_params.json"
            if fp.exists():
                data = json.loads(fp.read_text(encoding="utf-8"))
                bp = data["best_params"]
                sup_params[(event, budget, seed)] = {
                    "lr": bp["lr"],
                    "batch_size": bp["batch_size"],
                    "epochs": bp["max_epochs"],
                }

print(f"Loaded supervised Optuna params: {len(sup_params)} / {len(EVENTS) * len(BUDGETS) * len(SEEDS)}")
if len(sup_params) < len(EVENTS) * len(BUDGETS) * len(SEEDS):
    raise RuntimeError(
        "Missing supervised Optuna params at " + str(SUP_OPTUNA_DIR) + ".
"
        "These should be shipped under `results/bertweet/supervised/optuna-tuned/` "
        "(120 best_params.json files). If you removed them, run "
        "`python -m supervised_baseline.optuna_tuner --n-trials 10` to regenerate."
    )

In [ ]:
# Helper: generate pseudo-labels for a given budget
def generate_pseudo_labels(budgets_to_run):
    """Generate self-trained pseudo-labels using supervised Optuna params.
    
    Calls train_and_predict_one_cell directly but patches the output path
    to write to self-trained-optuna/ instead of self-trained/.
    """
    from lg_cotrain.generate_selftrained_teacher import train_and_predict_one_cell
    import multiprocessing as mp
    from concurrent.futures import ProcessPoolExecutor, as_completed

    pending = []
    skipped = 0
    for event in EVENTS:
        for budget in budgets_to_run:
            for seed in SEEDS:
                out_path = (
                    PSEUDO_LABEL_DIR / event
                    / f"labeled_{budget}_set{seed}_pseudo.csv"
                )
                if out_path.exists():
                    skipped += 1
                    continue
                hp = sup_params[(event, budget, seed)]
                pending.append(dict(
                    event=event, budget=budget, seed_set=seed,
                    data_root=DATA_ROOT,
                    model_name=MODEL_NAME,
                    lr=hp["lr"],
                    batch_size=hp["batch_size"],
                    epochs=hp["epochs"],
                ))

    print(f"Budgets: {budgets_to_run}")
    print(f"Skipped (exist): {skipped}, Pending: {len(pending)}")

    if not pending:
        print("All pseudo-labels already generated.")
        return

    start = time.time()

    def _run_one(kw, device_str):
        """Run teacher training, then move output to self-trained-optuna/."""
        import shutil
        kw_copy = dict(kw, device=device_str)
        r = train_and_predict_one_cell(**kw_copy)
        if r["status"] == "ok":
            # Move from default self-trained/ to self-trained-optuna/
            src = Path(r["output_path"])
            dst = (
                PSEUDO_LABEL_DIR / kw["event"]
                / f"labeled_{kw['budget']}_set{kw['seed_set']}_pseudo.csv"
            )
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            r["output_path"] = str(dst)
        return r

    for i, kw in enumerate(pending, 1):
        device_str = f"cuda:{(i - 1) % N_GPUS}"
        cell_name = f"{kw['event']}/{kw['budget']}_set{kw['seed_set']}"
        print(f"  [{i}/{len(pending)}] {cell_name} (lr={kw['lr']:.6f}, bs={kw['batch_size']}, ep={kw['epochs']})...", end="")
        r = _run_one(kw, device_str)
        print(f" {r['status']}")

    elapsed = time.time() - start
    print(f"\nWall clock: {elapsed:.0f}s = {elapsed/60:.1f}min")


# Helper: run Optuna for a given budget
def run_optuna(budgets_to_run):
    """Run LG-CoTrain Optuna per-experiment with self-trained-optuna pseudo-labels.
    
    Tunes same 6 params as LG-CoTrain Optuna: lr, batch_size, cotrain_epochs,
    finetune_patience, weight_decay, warmup_ratio.
    """
    cmd = [
        sys.executable, "-m", "lg_cotrain.optuna_per_experiment",
        "--n-trials", str(N_OPTUNA_TRIALS),
        "--num-gpus", str(N_GPUS),
        "--events", *EVENTS,
        "--budgets", *(str(b) for b in budgets_to_run),
        "--seed-sets", *(str(s) for s in SEEDS),
        "--pseudo-label-source", "self-trained-optuna",
        "--model-name", MODEL_NAME,
        "--data-root", str(DATA_ROOT),
        "--storage-dir", str(OPTUNA_STORAGE),
    ]
    n_studies = len(EVENTS) * len(budgets_to_run) * len(SEEDS)
    print(f"Budgets: {budgets_to_run}")
    print(f"Expected: {n_studies} studies x {N_OPTUNA_TRIALS} trials = {n_studies * N_OPTUNA_TRIALS} runs")
    print("Command:")
    print("  " + " ".join(cmd))
    print("=" * 70)

    start = time.time()
    proc = subprocess.run(cmd, cwd=str(repo_root))
    elapsed = time.time() - start

    print("=" * 70)
    print(f"Wall clock: {elapsed:.0f}s = {elapsed/3600:.2f}h")
    if proc.returncode != 0:
        print(f"WARNING: exit code {proc.returncode}")

    n_done = len(list(OPTUNA_STORAGE.glob(
        f"*/*/trials_{N_OPTUNA_TRIALS}/best_params.json"
    )))
    total_expected = len(EVENTS) * len(BUDGETS) * len(SEEDS)
    print(f"\nbest_params.json files: {n_done} / {total_expected} (all budgets)")


# Helper: analyze results and compare
def analyze(budgets_to_show=None):
    """Compare self-trained Optuna vs LG-CoTrain Optuna vs supervised Optuna."""
    if budgets_to_show is None:
        budgets_to_show = BUDGETS

    # Load self-trained Optuna results
    st = defaultdict(list)
    for f in sorted(OPTUNA_STORAGE.glob(f"*/*/trials_{N_OPTUNA_TRIALS}/best_params.json")):
        p = json.loads(f.read_text(encoding="utf-8"))
        fm = p.get("best_full_metrics", {})
        if not fm:
            continue
        budget = fm["budget"]
        if budget in budgets_to_show:
            st[budget].append(fm["test_macro_f1"])

    # Load LG-CoTrain Optuna (GPT-4o) results
    lg = defaultdict(list)
    for f in sorted(LG_COTRAIN_OPTUNA.glob(f"*/*/trials_{N_OPTUNA_TRIALS}/best_params.json")):
        p = json.loads(f.read_text(encoding="utf-8"))
        fm = p.get("best_full_metrics", {})
        if not fm:
            continue
        budget = fm["budget"]
        if budget in budgets_to_show:
            lg[budget].append(fm["test_macro_f1"])

    # Load supervised Optuna results
    sup = defaultdict(list)
    for f in sorted(SUP_OPTUNA_DIR.glob(f"*/*/trials_{N_OPTUNA_TRIALS}/best_params.json")):
        p = json.loads(f.read_text(encoding="utf-8"))
        fm = p.get("best_full_metrics", {})
        if not fm:
            continue
        budget = fm["budget"]
        if budget in budgets_to_show:
            sup[budget].append(fm["test_macro_f1"])

    print("=" * 95)
    print("COMPARISON: Self-trained Optuna vs LG-CoTrain Optuna vs Supervised Optuna")
    print("LG-CoTrain + Self-trained: 6 search params | Supervised: 3 search params")
    print("=" * 95)
    print(f'{"Budget":>8s}  {"LG-CoTrain (GPT-4o)":>22s}  {"Self-trained (optuna)":>22s}  {"Supervised":>22s}  {"Delta (ST-LG)":>14s}')
    print("-" * 95)

    for b in budgets_to_show:
        lg_vals = lg.get(b, [])
        st_vals = st.get(b, [])
        sup_vals = sup.get(b, [])

        lg_str = f"{sum(lg_vals)/len(lg_vals):.4f} (n={len(lg_vals)})" if lg_vals else "n/a"
        st_str = f"{sum(st_vals)/len(st_vals):.4f} (n={len(st_vals)})" if st_vals else "n/a"
        sup_str = f"{sum(sup_vals)/len(sup_vals):.4f} (n={len(sup_vals)})" if sup_vals else "n/a"

        if lg_vals and st_vals:
            delta = sum(st_vals)/len(st_vals) - sum(lg_vals)/len(lg_vals)
            delta_str = f"{delta:+.4f}"
        else:
            delta_str = "n/a"

        print(f"{b:>8d}  {lg_str:>22s}  {st_str:>22s}  {sup_str:>22s}  {delta_str:>14s}")

    # Per-event breakdown for completed budgets
    print()
    for b in budgets_to_show:
        st_by_event = defaultdict(list)
        lg_by_event = defaultdict(list)
        for f in sorted(OPTUNA_STORAGE.glob(f"*/{b}_set*/trials_{N_OPTUNA_TRIALS}/best_params.json")):
            p = json.loads(f.read_text(encoding="utf-8"))
            fm = p.get("best_full_metrics", {})
            if fm:
                st_by_event[fm["event"]].append(fm["test_macro_f1"])
        for f in sorted(LG_COTRAIN_OPTUNA.glob(f"*/{b}_set*/trials_{N_OPTUNA_TRIALS}/best_params.json")):
            p = json.loads(f.read_text(encoding="utf-8"))
            fm = p.get("best_full_metrics", {})
            if fm:
                lg_by_event[fm["event"]].append(fm["test_macro_f1"])

        if not st_by_event:
            continue

        print(f"--- Budget {b} per-event ---")
        print(f'{"Event":<35s}  {"LG-CoTrain":>10s}  {"Self-trained":>12s}  {"Delta":>8s}')
        for event in EVENTS:
            lg_v = lg_by_event.get(event, [])
            st_v = st_by_event.get(event, [])
            lg_m = sum(lg_v)/len(lg_v) if lg_v else 0
            st_m = sum(st_v)/len(st_v) if st_v else 0
            lg_s = f"{lg_m:.4f}" if lg_v else "n/a"
            st_s = f"{st_m:.4f}" if st_v else "n/a"
            d_s = f"{st_m - lg_m:+.4f}" if lg_v and st_v else "n/a"
            print(f"{event:<35s}  {lg_s:>10s}  {st_s:>12s}  {d_s:>8s}")
        print()

## Budget 5: Generate pseudo-labels + Optuna

30 teacher trainings (fast) + 300 LG-CoTrain Optuna trials.

In [ ]:
generate_pseudo_labels(budgets_to_run=[5])

In [ ]:
run_optuna(budgets_to_run=[5])

In [ ]:
# Standalone analyze - reads directly from results folder.
# Can run independently without having executed earlier cells.
import json
from pathlib import Path
from collections import defaultdict

def _find_repo():
    for c in [Path().resolve()] + list(Path().resolve().parents):
        if (c / "lg_cotrain").is_dir():
            return c
    raise RuntimeError("Cannot find repo root")
_repo = _find_repo()
_N_TRIALS = 10
_BUDGETS_TO_SHOW = [5]
_EVENTS = [
    "kaikoura_earthquake_2016", "canada_wildfires_2016", "cyclone_idai_2019",
    "hurricane_florence_2018", "hurricane_maria_2017", "california_wildfires_2018",
    "hurricane_dorian_2019", "kerala_floods_2018", "hurricane_harvey_2017",
    "hurricane_irma_2017",
]
_SOURCES = {
    "LG-CoTrain (GPT-4o)": _repo / "results" / "bertweet" / "optuna" / "wge7",
    "Self-trained (optuna)": _repo / "results" / "bertweet" / "ablation" / "self-trained" / "optuna",
    "Supervised": _repo / "results" / "bertweet" / "supervised" / "optuna-tuned",
}


def _load(root, budgets_filter=None, by_event_budget=None):
    data = defaultdict(list)
    for f in sorted(Path(root).glob(f"*/*/trials_{_N_TRIALS}/best_params.json")):
        p = json.loads(f.read_text(encoding="utf-8"))
        fm = p.get("best_full_metrics", {})
        if not fm:
            continue
        b = fm.get("budget")
        if budgets_filter is not None and b not in budgets_filter:
            continue
        if by_event_budget is not None:
            if b != by_event_budget:
                continue
            data[fm["event"]].append(fm["test_macro_f1"])
        else:
            data[b].append(fm["test_macro_f1"])
    return data


# Per-budget summary
by_budget = {label: _load(p, _BUDGETS_TO_SHOW) for label, p in _SOURCES.items()}

print("=" * 95)
print("COMPARISON: Self-trained Optuna vs LG-CoTrain Optuna vs Supervised Optuna")
print("=" * 95)
print(f'{"Budget":>8s}  {"LG-CoTrain (GPT-4o)":>22s}  {"Self-trained (optuna)":>22s}  {"Supervised":>22s}  {"Delta (ST-LG)":>14s}')
print("-" * 95)
for b in _BUDGETS_TO_SHOW:
    lg_v = by_budget["LG-CoTrain (GPT-4o)"].get(b, [])
    st_v = by_budget["Self-trained (optuna)"].get(b, [])
    sup_v = by_budget["Supervised"].get(b, [])
    lg_s = f"{sum(lg_v)/len(lg_v):.4f} (n={len(lg_v)})" if lg_v else "n/a"
    st_s = f"{sum(st_v)/len(st_v):.4f} (n={len(st_v)})" if st_v else "n/a"
    sup_s = f"{sum(sup_v)/len(sup_v):.4f} (n={len(sup_v)})" if sup_v else "n/a"
    d_s = f"{sum(st_v)/len(st_v) - sum(lg_v)/len(lg_v):+.4f}" if lg_v and st_v else "n/a"
    print(f"{b:>8d}  {lg_s:>22s}  {st_s:>22s}  {sup_s:>22s}  {d_s:>14s}")

# Per-event breakdown
print()
for b in _BUDGETS_TO_SHOW:
    st_by = _load(_SOURCES["Self-trained (optuna)"], by_event_budget=b)
    lg_by = _load(_SOURCES["LG-CoTrain (GPT-4o)"], by_event_budget=b)
    sup_by = _load(_SOURCES["Supervised"], by_event_budget=b)
    if not st_by:
        continue
    print(f"--- Budget {b} per-event ---")
    print(f'{"Event":<30s}  {"LG-CoTrain":>10s}  {"Self-trained":>12s}  {"Supervised":>10s}  {"Delta":>8s}  {"Seeds":>6s}')
    print("-" * 80)
    for event in _EVENTS:
        lg_v, st_v, sup_v = lg_by.get(event, []), st_by.get(event, []), sup_by.get(event, [])
        lg_m = sum(lg_v)/len(lg_v) if lg_v else 0
        st_m = sum(st_v)/len(st_v) if st_v else 0
        sup_m = sum(sup_v)/len(sup_v) if sup_v else 0
        lg_s = f"{lg_m:.4f}" if lg_v else "n/a"
        st_s = f"{st_m:.4f}" if st_v else "n/a"
        sup_s = f"{sup_m:.4f}" if sup_v else "n/a"
        d_s = f"{st_m - lg_m:+.4f}" if lg_v and st_v else "n/a"
        seeds = f"{len(st_v)}/3" if st_v else "0/3"
        print(f"{event:<30s}  {lg_s:>10s}  {st_s:>12s}  {sup_s:>10s}  {d_s:>8s}  {seeds:>6s}")
    print()

## Budget 10: Generate pseudo-labels + Optuna

In [ ]:
generate_pseudo_labels(budgets_to_run=[10])

In [ ]:
run_optuna(budgets_to_run=[10])

In [ ]:
# Standalone analyze - reads directly from results folder.
# Can run independently without having executed earlier cells.
import json
from pathlib import Path
from collections import defaultdict

def _find_repo():
    for c in [Path().resolve()] + list(Path().resolve().parents):
        if (c / "lg_cotrain").is_dir():
            return c
    raise RuntimeError("Cannot find repo root")
_repo = _find_repo()
_N_TRIALS = 10
_BUDGETS_TO_SHOW = [5, 10]
_EVENTS = [
    "kaikoura_earthquake_2016", "canada_wildfires_2016", "cyclone_idai_2019",
    "hurricane_florence_2018", "hurricane_maria_2017", "california_wildfires_2018",
    "hurricane_dorian_2019", "kerala_floods_2018", "hurricane_harvey_2017",
    "hurricane_irma_2017",
]
_SOURCES = {
    "LG-CoTrain (GPT-4o)": _repo / "results" / "bertweet" / "optuna" / "wge7",
    "Self-trained (optuna)": _repo / "results" / "bertweet" / "ablation" / "self-trained" / "optuna",
    "Supervised": _repo / "results" / "bertweet" / "supervised" / "optuna-tuned",
}


def _load(root, budgets_filter=None, by_event_budget=None):
    data = defaultdict(list)
    for f in sorted(Path(root).glob(f"*/*/trials_{_N_TRIALS}/best_params.json")):
        p = json.loads(f.read_text(encoding="utf-8"))
        fm = p.get("best_full_metrics", {})
        if not fm:
            continue
        b = fm.get("budget")
        if budgets_filter is not None and b not in budgets_filter:
            continue
        if by_event_budget is not None:
            if b != by_event_budget:
                continue
            data[fm["event"]].append(fm["test_macro_f1"])
        else:
            data[b].append(fm["test_macro_f1"])
    return data


by_budget = {label: _load(p, _BUDGETS_TO_SHOW) for label, p in _SOURCES.items()}

print("=" * 95)
print("COMPARISON: Self-trained Optuna vs LG-CoTrain Optuna vs Supervised Optuna")
print("=" * 95)
print(f'{"Budget":>8s}  {"LG-CoTrain (GPT-4o)":>22s}  {"Self-trained (optuna)":>22s}  {"Supervised":>22s}  {"Delta (ST-LG)":>14s}')
print("-" * 95)
for b in _BUDGETS_TO_SHOW:
    lg_v = by_budget["LG-CoTrain (GPT-4o)"].get(b, [])
    st_v = by_budget["Self-trained (optuna)"].get(b, [])
    sup_v = by_budget["Supervised"].get(b, [])
    lg_s = f"{sum(lg_v)/len(lg_v):.4f} (n={len(lg_v)})" if lg_v else "n/a"
    st_s = f"{sum(st_v)/len(st_v):.4f} (n={len(st_v)})" if st_v else "n/a"
    sup_s = f"{sum(sup_v)/len(sup_v):.4f} (n={len(sup_v)})" if sup_v else "n/a"
    d_s = f"{sum(st_v)/len(st_v) - sum(lg_v)/len(lg_v):+.4f}" if lg_v and st_v else "n/a"
    print(f"{b:>8d}  {lg_s:>22s}  {st_s:>22s}  {sup_s:>22s}  {d_s:>14s}")

print()
for b in _BUDGETS_TO_SHOW:
    st_by = _load(_SOURCES["Self-trained (optuna)"], by_event_budget=b)
    lg_by = _load(_SOURCES["LG-CoTrain (GPT-4o)"], by_event_budget=b)
    sup_by = _load(_SOURCES["Supervised"], by_event_budget=b)
    if not st_by:
        continue
    print(f"--- Budget {b} per-event ---")
    print(f'{"Event":<30s}  {"LG-CoTrain":>10s}  {"Self-trained":>12s}  {"Supervised":>10s}  {"Delta":>8s}  {"Seeds":>6s}')
    print("-" * 80)
    for event in _EVENTS:
        lg_v, st_v, sup_v = lg_by.get(event, []), st_by.get(event, []), sup_by.get(event, [])
        lg_m = sum(lg_v)/len(lg_v) if lg_v else 0
        st_m = sum(st_v)/len(st_v) if st_v else 0
        sup_m = sum(sup_v)/len(sup_v) if sup_v else 0
        lg_s = f"{lg_m:.4f}" if lg_v else "n/a"
        st_s = f"{st_m:.4f}" if st_v else "n/a"
        sup_s = f"{sup_m:.4f}" if sup_v else "n/a"
        d_s = f"{st_m - lg_m:+.4f}" if lg_v and st_v else "n/a"
        seeds = f"{len(st_v)}/3" if st_v else "0/3"
        print(f"{event:<30s}  {lg_s:>10s}  {st_s:>12s}  {sup_s:>10s}  {d_s:>8s}  {seeds:>6s}")
    print()

## Budget 25: Generate pseudo-labels + Optuna

In [ ]:
generate_pseudo_labels(budgets_to_run=[25])

In [ ]:
run_optuna(budgets_to_run=[25])

In [ ]:
# Standalone analyze - reads directly from results folder.
import json
from pathlib import Path
from collections import defaultdict

def _find_repo():
    for c in [Path().resolve()] + list(Path().resolve().parents):
        if (c / "lg_cotrain").is_dir():
            return c
    raise RuntimeError("Cannot find repo root")
_repo = _find_repo()
_N_TRIALS = 10
_BUDGETS_TO_SHOW = [5, 10, 25]
_EVENTS = [
    "kaikoura_earthquake_2016", "canada_wildfires_2016", "cyclone_idai_2019",
    "hurricane_florence_2018", "hurricane_maria_2017", "california_wildfires_2018",
    "hurricane_dorian_2019", "kerala_floods_2018", "hurricane_harvey_2017",
    "hurricane_irma_2017",
]
_SOURCES = {
    "LG-CoTrain (GPT-4o)": _repo / "results" / "bertweet" / "optuna" / "wge7",
    "Self-trained (optuna)": _repo / "results" / "bertweet" / "ablation" / "self-trained" / "optuna",
    "Supervised": _repo / "results" / "bertweet" / "supervised" / "optuna-tuned",
}


def _load(root, budgets_filter=None, by_event_budget=None):
    data = defaultdict(list)
    for f in sorted(Path(root).glob(f"*/*/trials_{_N_TRIALS}/best_params.json")):
        p = json.loads(f.read_text(encoding="utf-8"))
        fm = p.get("best_full_metrics", {})
        if not fm:
            continue
        b = fm.get("budget")
        if budgets_filter is not None and b not in budgets_filter:
            continue
        if by_event_budget is not None:
            if b != by_event_budget:
                continue
            data[fm["event"]].append(fm["test_macro_f1"])
        else:
            data[b].append(fm["test_macro_f1"])
    return data


by_budget = {label: _load(p, _BUDGETS_TO_SHOW) for label, p in _SOURCES.items()}

print("=" * 95)
print("COMPARISON: Self-trained Optuna vs LG-CoTrain Optuna vs Supervised Optuna")
print("=" * 95)
print(f'{"Budget":>8s}  {"LG-CoTrain (GPT-4o)":>22s}  {"Self-trained (optuna)":>22s}  {"Supervised":>22s}  {"Delta (ST-LG)":>14s}')
print("-" * 95)
for b in _BUDGETS_TO_SHOW:
    lg_v = by_budget["LG-CoTrain (GPT-4o)"].get(b, [])
    st_v = by_budget["Self-trained (optuna)"].get(b, [])
    sup_v = by_budget["Supervised"].get(b, [])
    lg_s = f"{sum(lg_v)/len(lg_v):.4f} (n={len(lg_v)})" if lg_v else "n/a"
    st_s = f"{sum(st_v)/len(st_v):.4f} (n={len(st_v)})" if st_v else "n/a"
    sup_s = f"{sum(sup_v)/len(sup_v):.4f} (n={len(sup_v)})" if sup_v else "n/a"
    d_s = f"{sum(st_v)/len(st_v) - sum(lg_v)/len(lg_v):+.4f}" if lg_v and st_v else "n/a"
    print(f"{b:>8d}  {lg_s:>22s}  {st_s:>22s}  {sup_s:>22s}  {d_s:>14s}")

print()
for b in _BUDGETS_TO_SHOW:
    st_by = _load(_SOURCES["Self-trained (optuna)"], by_event_budget=b)
    lg_by = _load(_SOURCES["LG-CoTrain (GPT-4o)"], by_event_budget=b)
    sup_by = _load(_SOURCES["Supervised"], by_event_budget=b)
    if not st_by:
        continue
    print(f"--- Budget {b} per-event ---")
    print(f'{"Event":<30s}  {"LG-CoTrain":>10s}  {"Self-trained":>12s}  {"Supervised":>10s}  {"Delta":>8s}  {"Seeds":>6s}')
    print("-" * 80)
    for event in _EVENTS:
        lg_v, st_v, sup_v = lg_by.get(event, []), st_by.get(event, []), sup_by.get(event, [])
        lg_m = sum(lg_v)/len(lg_v) if lg_v else 0
        st_m = sum(st_v)/len(st_v) if st_v else 0
        sup_m = sum(sup_v)/len(sup_v) if sup_v else 0
        lg_s = f"{lg_m:.4f}" if lg_v else "n/a"
        st_s = f"{st_m:.4f}" if st_v else "n/a"
        sup_s = f"{sup_m:.4f}" if sup_v else "n/a"
        d_s = f"{st_m - lg_m:+.4f}" if lg_v and st_v else "n/a"
        seeds = f"{len(st_v)}/3" if st_v else "0/3"
        print(f"{event:<30s}  {lg_s:>10s}  {st_s:>12s}  {sup_s:>10s}  {d_s:>8s}  {seeds:>6s}")
    print()

## Budget 50: Generate pseudo-labels + Optuna

In [ ]:
generate_pseudo_labels(budgets_to_run=[50])

In [ ]:
run_optuna(budgets_to_run=[50])

In [ ]:
# Standalone analyze - reads directly from results folder.
import json
from pathlib import Path
from collections import defaultdict

def _find_repo():
    for c in [Path().resolve()] + list(Path().resolve().parents):
        if (c / "lg_cotrain").is_dir():
            return c
    raise RuntimeError("Cannot find repo root")
_repo = _find_repo()
_N_TRIALS = 10
_BUDGETS_TO_SHOW = [5, 10, 25, 50]
_EVENTS = [
    "kaikoura_earthquake_2016", "canada_wildfires_2016", "cyclone_idai_2019",
    "hurricane_florence_2018", "hurricane_maria_2017", "california_wildfires_2018",
    "hurricane_dorian_2019", "kerala_floods_2018", "hurricane_harvey_2017",
    "hurricane_irma_2017",
]
_SOURCES = {
    "LG-CoTrain (GPT-4o)": _repo / "results" / "bertweet" / "optuna" / "wge7",
    "Self-trained (optuna)": _repo / "results" / "bertweet" / "ablation" / "self-trained" / "optuna",
    "Supervised": _repo / "results" / "bertweet" / "supervised" / "optuna-tuned",
}


def _load(root, budgets_filter=None, by_event_budget=None):
    data = defaultdict(list)
    for f in sorted(Path(root).glob(f"*/*/trials_{_N_TRIALS}/best_params.json")):
        p = json.loads(f.read_text(encoding="utf-8"))
        fm = p.get("best_full_metrics", {})
        if not fm:
            continue
        b = fm.get("budget")
        if budgets_filter is not None and b not in budgets_filter:
            continue
        if by_event_budget is not None:
            if b != by_event_budget:
                continue
            data[fm["event"]].append(fm["test_macro_f1"])
        else:
            data[b].append(fm["test_macro_f1"])
    return data


by_budget = {label: _load(p, _BUDGETS_TO_SHOW) for label, p in _SOURCES.items()}

print("=" * 95)
print("COMPARISON: Self-trained Optuna vs LG-CoTrain Optuna vs Supervised Optuna")
print("=" * 95)
print(f'{"Budget":>8s}  {"LG-CoTrain (GPT-4o)":>22s}  {"Self-trained (optuna)":>22s}  {"Supervised":>22s}  {"Delta (ST-LG)":>14s}')
print("-" * 95)
for b in _BUDGETS_TO_SHOW:
    lg_v = by_budget["LG-CoTrain (GPT-4o)"].get(b, [])
    st_v = by_budget["Self-trained (optuna)"].get(b, [])
    sup_v = by_budget["Supervised"].get(b, [])
    lg_s = f"{sum(lg_v)/len(lg_v):.4f} (n={len(lg_v)})" if lg_v else "n/a"
    st_s = f"{sum(st_v)/len(st_v):.4f} (n={len(st_v)})" if st_v else "n/a"
    sup_s = f"{sum(sup_v)/len(sup_v):.4f} (n={len(sup_v)})" if sup_v else "n/a"
    d_s = f"{sum(st_v)/len(st_v) - sum(lg_v)/len(lg_v):+.4f}" if lg_v and st_v else "n/a"
    print(f"{b:>8d}  {lg_s:>22s}  {st_s:>22s}  {sup_s:>22s}  {d_s:>14s}")

print()
for b in _BUDGETS_TO_SHOW:
    st_by = _load(_SOURCES["Self-trained (optuna)"], by_event_budget=b)
    lg_by = _load(_SOURCES["LG-CoTrain (GPT-4o)"], by_event_budget=b)
    sup_by = _load(_SOURCES["Supervised"], by_event_budget=b)
    if not st_by:
        continue
    print(f"--- Budget {b} per-event ---")
    print(f'{"Event":<30s}  {"LG-CoTrain":>10s}  {"Self-trained":>12s}  {"Supervised":>10s}  {"Delta":>8s}  {"Seeds":>6s}')
    print("-" * 80)
    for event in _EVENTS:
        lg_v, st_v, sup_v = lg_by.get(event, []), st_by.get(event, []), sup_by.get(event, [])
        lg_m = sum(lg_v)/len(lg_v) if lg_v else 0
        st_m = sum(st_v)/len(st_v) if st_v else 0
        sup_m = sum(sup_v)/len(sup_v) if sup_v else 0
        lg_s = f"{lg_m:.4f}" if lg_v else "n/a"
        st_s = f"{st_m:.4f}" if st_v else "n/a"
        sup_s = f"{sup_m:.4f}" if sup_v else "n/a"
        d_s = f"{st_m - lg_m:+.4f}" if lg_v and st_v else "n/a"
        seeds = f"{len(st_v)}/3" if st_v else "0/3"
        print(f"{event:<30s}  {lg_s:>10s}  {st_s:>12s}  {sup_s:>10s}  {d_s:>8s}  {seeds:>6s}")
    print()

## Summary

This notebook produces a fair comparison between self-trained and GPT-4o
pseudo-labels, with both using full per-experiment Optuna tuning (same
6-parameter search space, same number of trials). The only variable is
the pseudo-label source.